In [1]:
# 🔹 1. Load .env
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("❌ GEMINI_API_KEY not found in .env file")


# 🔹 2. Setup config_list
config_list = [
    {
        "model": "gemini-2.5-flash",
        "api_key": api_key,
        "api_type": "google"
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0.5,
}


# 🔹 3. Create Assistant
from autogen import AssistantAgent

assistant = AssistantAgent(
    name="assistant",
    llm_config=llm_config
)


# 🔹 4. Load docs_content (PDF → TEXT)
from PyPDF2 import PdfReader

def extract_pdf_text(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

# 👉 Change path if needed
pdf_path = "autogen-docs/book.pdf"

docs_content = extract_pdf_text(pdf_path)

# 🔥 IMPORTANT: limit size (prevents crash)
docs_content = docs_content[:5000]


# 🔹 5. Create RAG Agent (NO CHROMA)
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_content": docs_content,
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": None   # 🔥 avoids torch/chroma error
    },
    code_execution_config=False,
)


# 🔹 6. Run chat
assistant.reset()

qa_problem = "Summarize the content and perform any calculations if present"

chat_result = ragproxyagent.initiate_chat(
    assistant,
    message=qa_problem
)

print(chat_result)

2026-04-05 17:08:06,411 - autogen.agentchat.contrib.retrieve_user_proxy_agent - WARNING - docs_path is not provided in retrieve_config. Will raise ValueError if the collection `ag2-docs` doesn't exist. Set docs_path to None to suppress this warning.


ragproxyagent (to assistant):

Summarize the content and perform any calculations if present

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Please provide the content you would like me to summarize and from which I should perform calculations.

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (42fa7b03-489d-4671-a255-8671f3c7b7b7): Termination message condition on agent 'ragproxyagent' met
ChatResult(chat_id=120823042177538098128660794959050698825, chat_history=[{'content': 'Summarize the content and perform any calculations if present', 'role': 'assistant', 'name': 'ragproxyagent'}, {'content': 'Please provide the content you would like me to summarize and from which I should perform calculations.', 'role': 'user', 'name': 'assistant'}], summary='Please provide the content you would like me to summarize and from which I should perform calculations.', cost={'usage